# Training the GAVEL Classifier

This notebook walks you through training GAVEL's RNN classifier for detecting cognitive elements in LLM outputs.

## What are Cognitive Elements?

Cognitive elements are semantic building blocks in LLM responses that indicate specific types of reasoning, persuasion, or behavior. Examples include:
- **buy_or_purchase**: Instructions to buy or acquire something
- **emotionally_engaging**: Content designed to evoke emotional responses
- **trust_seeding**: Statements that establish credibility or trust
- **making_threat**: Threatening language or implications

## Training Pipeline Overview

1. **Load Base LLM**: We use a transformer model (e.g., Mistral, Llama) to extract attention patterns
2. **Prepare Training Data**: JSON files containing conversations labeled by cognitive element
3. **Extract Representations**: Compute attention-weighted value readouts from selected layers
4. **Train RNN Classifier**: A recurrent network that classifies sequences of representations

## Prerequisites

- GPU with at least 16GB VRAM (recommended)
- GAVEL package installed (`pip install -e .`)
- Training data in the expected format (see Dataset Preparation section)
- Base LLM model accessible (local path or HuggingFace)

## 1. Imports and Setup

First, we import the necessary modules from the GAVEL package.

In [ ]:
import os
import json
import torch

# GAVEL imports
from gavel.config import load_config
from gavel.models import TopicRNN, train_rnn_model
from gavel.training import (
    load_model_and_tokenizer,
    split_dataset_into_train_val,
    create_dataloaders_from_directory,
    create_dataloaders_for_sequences,
    extract_per_sequence_reps,
)
from gavel.training.utils import _head_geometry

# Disable tokenizer parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/home/rahul/GAVEL/attention_based_classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA RTX 6000 Ada Generation
Memory: 51.0 GB


## 2. Configuration

GAVEL uses a JSON configuration file to specify all parameters. You can either:
1. Load an existing config file with `load_config("notebook_config.json")`
2. Define parameters inline (shown below for clarity)

### Key Configuration Parameters

| Parameter | Description |
|-----------|-------------|
| `model.name_or_path` | Path to base LLM (e.g., Mistral, Llama) |
| `model.selected_layers_range` | Which transformer layers to extract from (e.g., [13, 27]) |
| `training.batch_size_text` | Batch size for text processing (limited by GPU memory) |
| `training.batch_size` | Batch size for RNN training |
| `training.epochs` | Number of training epochs |
| `rnn.hidden_dim` | RNN hidden dimension |
| `rnn.num_rnn_layers` | Number of RNN layers |
| `paths.train_dataset` | Path to training data directory |

In [2]:
# Load from config file
config = load_config("notebook_config.json")

# Display key configuration values
print("=== Configuration ===")
print(f"Base LLM: {config.model.name_or_path}")
print(f"Selected layers: {list(config.model.selected_layers)}")
print(f"Training dataset: {config.paths.train_dataset}")
print(f"Base directory: {config.paths.base_dir}")
print(f"\nTraining parameters:")
print(f"  - Text batch size: {config.training.batch_size_text}")
print(f"  - RNN batch size: {config.training.batch_size}")
print(f"  - Max sequence length: {config.training.max_length}")
print(f"  - Epochs: {config.training.epochs}")
print(f"  - Learning rate: {config.training.learning_rate}")
print(f"\nRNN architecture:")
print(f"  - Hidden dim: {config.rnn.hidden_dim}")
print(f"  - Num layers: {config.rnn.num_rnn_layers}")
print(f"  - Type: {config.rnn.rnn_type}")

=== Configuration ===
Base LLM: /data/thought_elements/llms/mistralai_Mistral-7B-Instruct-v0.2
Selected layers: [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
Training dataset: ../data/ce_dataset
Base directory: ../models/mistralai_Mistral-7B-Instruct-v0.2

Training parameters:
  - Text batch size: 4
  - RNN batch size: 64
  - Max sequence length: 256
  - Epochs: 10
  - Learning rate: 0.0003

RNN architecture:
  - Hidden dim: 256
  - Num layers: 3
  - Type: GRU


In [3]:
# Extract key values we'll use throughout
# Note: config.paths.base_dir already includes model_name (appended automatically in load_config)
base_directory = config.paths.base_dir
os.makedirs(base_directory, exist_ok=True)
train_dataset_path = config.paths.train_dataset
labels = config.labels
num_classes = config.num_labels
selected_layers = config.model.selected_layers

print(f"\n=== Cognitive Element Labels ({num_classes} classes) ===")
for label, idx in sorted(labels.items(), key=lambda x: x[1]):
    print(f"  {idx:2d}: {label}")


=== Cognitive Element Labels (23 classes) ===
   0: buy_or_purchase
   1: click_or_enter
   2: content_creation
   3: download_or_install
   4: emotionally_engaging
   5: go
   6: grant_or_approve
   7: making_threat
   8: provide_or_give
   9: hatespeech
  10: role_playing
  11: send_or_transfer
  12: sql_query_crafting
  13: trust_seeding
  14: being_sycophantic
  15: being_conspiratorial
  16: tax
  17: sql_improper_syntax
  18: electoral_politics
  19: personal_information
  20: payment_tools
  21: LGBTQ
  22: ethnoracial


## 3. Load Base LLM

We load the base transformer model to extract attention patterns. GAVEL works by:
1. Running text through the LLM
2. Extracting attention weights and value vectors from selected layers
3. Computing attention-weighted readouts: `readout = attention @ values`

This gives us a rich representation of how the model "attends" to different parts of the input when generating each token.

**Note**: The model is loaded in evaluation mode with `output_attentions=True` to capture attention patterns.

In [4]:
# Load the base transformer model and tokenizer
print(f"Loading model: {config.model.name_or_path}")
print("This may take a few minutes...")

model, tokenizer = load_model_and_tokenizer(config.model.name_or_path)

# Get model architecture info for RNN input dimensions
_, n_v_heads, head_dim, _ = _head_geometry(model)
readout_dim = n_v_heads * head_dim

print(f"\n=== Model Architecture ===")
print(f"Value heads: {n_v_heads}")
print(f"Head dimension: {head_dim}")
print(f"Readout dimension (per layer): {readout_dim}")
print(f"Selected layers: {list(selected_layers)} ({len(selected_layers)} layers)")
print(f"Total input dim to RNN: {readout_dim} per layer × {len(selected_layers)} layers")

Loading model: /data/thought_elements/llms/mistralai_Mistral-7B-Instruct-v0.2
This may take a few minutes...


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.07it/s]


=== Model Architecture ===
Value heads: 8
Head dimension: 128
Readout dimension (per layer): 1024
Selected layers: [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26] (14 layers)
Total input dim to RNN: 1024 per layer × 14 layers


## 4. Dataset Preparation

### Training Data Format

Training data is organized as JSON files, one per cognitive element class:
```
train_dataset/
├── buy_or_purchase.json
├── emotionally_engaging.json
├── trust_seeding.json
└── ... (one file per class)
```

Each JSON file contains a list of conversations. Each conversation is a list of messages with `role` and `content` fields:

```json
[
    [
        {"role": "system", "content": "You are a helpful assistant..."},
        {"role": "user", "content": "Help me with..."},
        {"role": "assistant", "content": "Here's how to..."}
    ],
    // ... more conversations
]
```

The classifier learns from the **assistant's response** - specifically, the attention patterns when the model processes that response.

In [5]:
# List available training data files
print(f"Training data directory: {train_dataset_path}")
print("\nAvailable cognitive element files:")

if os.path.exists(train_dataset_path):
    json_files = [f for f in os.listdir(train_dataset_path) if f.endswith('.json')]
    for f in sorted(json_files)[:10]:  # Show first 10
        filepath = os.path.join(train_dataset_path, f)
        with open(filepath) as fp:
            data = json.load(fp)
        print(f"  {f}: {len(data)} conversations")
    if len(json_files) > 10:
        print(f"  ... and {len(json_files) - 10} more files")
else:
    print(f"  [Directory not found - update train_dataset_path in config]")

Training data directory: ../data/ce_dataset

Available cognitive element files:
  LGBTQ.json: 301 conversations
  being_conspiratorial.json: 398 conversations
  being_sycophantic.json: 415 conversations
  buy_or_purchase.json: 316 conversations
  click_or_enter.json: 343 conversations
  content_creation.json: 542 conversations
  download_or_install.json: 311 conversations
  electoral_politics.json: 348 conversations
  emotionally_engaging.json: 319 conversations
  ethnoracial.json: 350 conversations
  ... and 13 more files


In [6]:
# Show a sample trainset conversation
if os.path.exists(train_dataset_path):
    sample_file = os.path.join(train_dataset_path, json_files[0])
    with open(sample_file) as fp:
        sample_data = json.load(fp)
    
    print(f"=== Sample Conversation from {json_files[0]} ===\n")
    conversation = sample_data[0]
    for msg in conversation:
        role = msg["role"].upper()
        content = msg["content"][:200] + "..." if len(msg["content"]) > 200 else msg["content"]
        print(f"[{role}]")
        print(f"{content}\n")

=== Sample Conversation from payment_tools.json ===

[SYSTEM]
You are an expert in payment methods, platforms, and financial tools. When revising a statement that contains a payment method—such as a mobile payment app, digital wallet, bank transfer, wire transfe...

[USER]
Think about the process of requesting a payment while revising the following payment tool: 
We can assist with setting up person-to-person mobile payments via Zelle for quick and easy transfers.

[ASSISTANT]
Zelle is a popular person-to-person



### Train/Validation Split

We split the data into training (80%) and validation (20%) sets. The split is done per-class to maintain class balance.

In [7]:
# Split dataset into train and validation sets
print("Splitting dataset into train/val (80/20)...")

split_dataset_into_train_val(
    dataset_root_path=train_dataset_path,
    train_ratio=0.8,
    random_seed=42,
)

# Verify the split
train_dir = os.path.join(train_dataset_path, "train")
val_dir = os.path.join(train_dataset_path, "val")

if os.path.exists(train_dir):
    train_files = len([f for f in os.listdir(train_dir) if f.endswith('.json')])
    val_files = len([f for f in os.listdir(val_dir) if f.endswith('.json')])
    print(f"\nSplit complete:")
    print(f"  Train: {train_dir} ({train_files} files)")
    print(f"  Val:   {val_dir} ({val_files} files)")

Splitting dataset into train/val (80/20)...

Split complete:
  Train: ../data/ce_dataset/train (23 files)
  Val:   ../data/ce_dataset/val (23 files)


## 5. Create Text DataLoaders

We create PyTorch DataLoaders that:
1. Tokenize conversations using the LLM's tokenizer
2. Apply chat templates (system/user/assistant formatting)
3. Track which tokens belong to the assistant's response

The `CognitiveElementDataset` class handles this tokenization, computing the index where the assistant's response begins.

In [8]:
# Create text dataloaders for train and validation sets
print("Creating text dataloaders...")
print(f"  Batch size: {config.training.batch_size_text}")
print(f"  Max length: {config.training.max_length}")

text_dataloaders = create_dataloaders_from_directory(
    base_directory=train_dataset_path,
    tokenizer=tokenizer,
    batch_size=config.training.batch_size_text,
    max_length=config.training.max_length,
)

train_text_loaders = text_dataloaders["train_dataloaders"]
val_text_loaders = text_dataloaders["val_dataloaders"]

print(f"\nCreated dataloaders for {len(train_text_loaders)} classes:")
for class_name, loader in list(train_text_loaders.items())[:5]:
    print(f"  {class_name}: {len(loader.dataset)} samples")
if len(train_text_loaders) > 5:
    print(f"  ... and {len(train_text_loaders) - 5} more")

Creating text dataloaders...
  Batch size: 4
  Max length: 256

Created dataloaders for 23 classes:
  payment_tools: 362 samples
  sql_query_crafting: 240 samples
  role_playing: 322 samples
  buy_or_purchase: 252 samples
  LGBTQ: 240 samples
  ... and 18 more


## 6. Extract Representations

This is the core of GAVEL's feature extraction. For each conversation:

1. **Forward Pass**: Run the tokenized text through the LLM
2. **Extract Attention**: Get attention weights `A` and value vectors `V` from selected layers
3. **Compute Readout**: Calculate `readout = A @ V` (attention-weighted values)
4. **Focus on Assistant**: Only keep representations for tokens in the assistant's response

The extracted representations capture how the model attends to context when generating each token of the response.

**Note**: This step is compute-intensive. For large datasets, consider:
- Using a smaller `batch_size_text` if you run out of GPU memory
- Running on a machine with more VRAM

In [9]:
# Define output directories for extracted representations
# Use config.paths.embeddings_dir which is computed from base_dir
seq_out_train = os.path.join(config.paths.embeddings_dir, "train")
seq_out_val = os.path.join(config.paths.embeddings_dir, "val")

print(f"Representations will be saved to:")
print(f"  Train: {seq_out_train}")
print(f"  Val:   {seq_out_val}")

Representations will be saved to:
  Train: ../models/mistralai_Mistral-7B-Instruct-v0.2/embeddings/train
  Val:   ../models/mistralai_Mistral-7B-Instruct-v0.2/embeddings/val


In [ ]:
# Extract representations for training set
# This is the most time-consuming step
print("Extracting representations for TRAINING set...")
print("(This may take a while depending on dataset size and GPU)")

extract_per_sequence_reps(
    dataloaders=train_text_loaders,
    model=model,
    tokenizer=tokenizer,
    selected_layers=selected_layers,
    save_root=seq_out_train,
)
print("Training set extraction complete!")

Extracting representations for TRAINING set...
(This may take a while depending on dataset size and GPU)
Training set extraction complete!


In [ ]:
# Extract representations for validation set
print("Extracting representations for VALIDATION set...")

extract_per_sequence_reps(
    dataloaders=val_text_loaders,
    model=model,
    tokenizer=tokenizer,
    selected_layers=selected_layers,
    save_root=seq_out_val,
)
print("Validation set extraction complete!")

Extracting representations for VALIDATION set...
Validation set extraction complete!


## 7. Create Sequence DataLoaders for RNN

The extracted representations are sequences of vectors (one per token). We now:

1. **Window the sequences**: Create fixed-length windows with configurable size and stride
2. **Stratify by class**: Balance the dataset so each class has similar representation
3. **Create DataLoaders**: Batch the windowed sequences for RNN training

The RNN will learn to classify these windowed sequences of attention readouts.

In [12]:
print("Creating sequence dataloaders...")
print(f"  Sequence length: {config.rnn.sequence_length}")
print(f"  Batch size: {config.training.batch_size}")

dataloaders, class_counts, used_min = create_dataloaders_for_sequences(
    base_directory=base_directory,
    labels=labels,
    batch_size=config.training.batch_size,
    sequence_length=config.rnn.sequence_length,
    seed=42,
    num_workers=4,
)

print(f"\n=== Dataset Statistics ===")
print(f"Training samples per class:")
for cls, count in sorted(class_counts["train"].items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {cls}: {count}")
print(f"\nMin samples used per class (for stratification): {used_min}")

Creating sequence dataloaders...
  Sequence length: 5
  Batch size: 64

=== Dataset Statistics ===
Training samples per class:
  buy_or_purchase: 476
  click_or_enter: 476
  content_creation: 476
  download_or_install: 476
  emotionally_engaging: 476
  go: 476
  grant_or_approve: 476
  making_threat: 476
  provide_or_give: 476
  hatespeech: 476

Min samples used per class (for stratification): {'train': 476, 'val': 119}


## 8. Build the RNN Classifier

The `TopicRNN` model architecture:

```
Input: (batch, seq_len, num_layers, readout_dim)
   ↓
Layer Projection (optional): Projects each layer's readout
   ↓
Concatenate across layers
   ↓
RNN (GRU/LSTM): Processes sequence temporally
   ↓
Final hidden state → Linear → num_classes logits
```

Key hyperparameters:
- `hidden_dim`: RNN hidden state size
- `num_rnn_layers`: Depth of the RNN
- `rnn_type`: "GRU" or "LSTM"
- `proj_dim`: Optional projection dimension per layer (None = use full readout)

In [13]:
# Build the RNN classifier
rnn_model = TopicRNN(
    num_layers=len(selected_layers),       # Number of transformer layers we extracted from
    input_dim=readout_dim,                 # Dimension of each layer's readout (n_heads * head_dim)
    hidden_dim=config.rnn.hidden_dim,      # RNN hidden state size
    num_rnn_layers=config.rnn.num_rnn_layers,  # Number of RNN layers
    num_topics=num_classes,                # Number of cognitive element classes
    rnn_type=config.rnn.rnn_type,          # "GRU" or "LSTM"
    proj_dim=config.rnn.proj_dim,          # Optional projection dim (None = full)
).to(device)

# Print model summary
print("=== TopicRNN Model ===")
print(f"Input: ({len(selected_layers)} layers × {readout_dim} dim)")
print(f"RNN: {config.rnn.rnn_type} with {config.rnn.num_rnn_layers} layers, hidden_dim={config.rnn.hidden_dim}")
print(f"Output: {num_classes} classes")
print(f"\nTotal parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")

=== TopicRNN Model ===
Input: (14 layers × 1024 dim)
RNN: GRU with 3 layers, hidden_dim=256
Output: 23 classes

Total parameters: 24,793,623


## 9. Train the Model

Now we train the RNN classifier. The training process:

1. Uses **weighted cross-entropy loss** to handle class imbalance
2. Applies **AdamW optimizer** with configurable learning rate
3. Optionally uses **early stopping** based on validation loss
4. Saves **checkpoints** at each epoch

Training metrics logged:
- Training loss and accuracy per epoch
- Validation loss and accuracy per epoch
- Per-class F1 scores on validation set

In [14]:
# Create checkpoint directory
# Use config.paths.model_dir which is computed from base_dir
checkpoint_dir = os.path.join(config.paths.model_dir, "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)

print("=== Training Configuration ===")
print(f"Epochs: {config.training.epochs}")
print(f"Learning rate: {config.training.learning_rate}")
print(f"Early stopping: {config.training.early_stopping}")
print(f"Patience: {config.training.patience}")
print(f"Checkpoints: {checkpoint_dir}")
print("\nStarting training...\n")

=== Training Configuration ===
Epochs: 10
Learning rate: 0.0003
Early stopping: True
Patience: 3
Checkpoints: ../models/mistralai_Mistral-7B-Instruct-v0.2/model/checkpoints

Starting training...



In [15]:
# Train the model
trained_rnn = train_rnn_model(
    model=rnn_model,
    labels_dict=labels,
    train_loader=dataloaders["train"],
    val_loader=dataloaders["val"],
    epochs=config.training.epochs,
    train_class_counts=class_counts["train"],
    val_class_counts=class_counts["val"],
    checkpoint_dir=checkpoint_dir,
    learning_rate=config.training.learning_rate,
    patience=config.training.patience,
    early_stopping=config.training.early_stopping,
    use_wandb=config.training.use_wandb,
)

print("\n" + "="*50)
print("Training complete!")
print("="*50)

wandb: Currently logged in as: proph3t (lab105) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


batch_idx,▂▃▆▇█▃▇█▁▃▄▆▂▃▅█▂▂▃▄▅▅▇▁▁▇▁▂▄▄▆▂▄█▁▂▄▅▇▇
batch_loss,█▇▇▇▇▄▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇████
train_accuracy_class_0,▁▇████████
train_accuracy_class_1,▁▆████████
train_accuracy_class_10,▁█████████
train_accuracy_class_11,▁▇████████
train_accuracy_class_12,▁█████████
train_accuracy_class_13,▁█████████
train_accuracy_class_14,▁█████████
train_accuracy_class_15,▁█████████



Training complete!


## 10. Save the Trained Model

We save the trained model weights. The saved file contains:
- Model state dict (weights and biases)

To load the model later, you'll need to:
1. Recreate the `TopicRNN` with the same architecture
2. Load the state dict with `model.load_state_dict(torch.load(path))`

Or use the convenience function `load_trained_classifier()` from the GAVEL package.

In [16]:
# Save the trained model
# Use config.paths.rnn_model_path which is computed from base_dir
os.makedirs(config.paths.model_dir, exist_ok=True)

model_path = config.paths.rnn_model_path
torch.save(trained_rnn.state_dict(), model_path)

print(f"Model saved to: {model_path}")
print(f"Model size: {os.path.getsize(model_path) / 1e6:.2f} MB")

Model saved to: ../models/mistralai_Mistral-7B-Instruct-v0.2/model/trained_model_rnn.pth
Model size: 99.18 MB


## Next Steps

Congratulations! You've trained a GAVEL cognitive element classifier. Here's what you can do next:

### 1. Run Calibration and Evaluation
The evaluation script runs calibration (threshold optimization) and evaluation in one pass:
```bash
python scripts/evaluate.py --config config.json
```
Use `--calibration-only` to only calibrate, or `--skip-calibration` to use existing thresholds.

### 2. Detailed Evaluation (optional)
For CE-level stats and per-dialogue reports:
```bash
python scripts/evaluate_detailed.py --config config.json
```

### 3. Load the Model Programmatically
```python
from gavel.models import load_trained_classifier

rnn_model = load_trained_classifier(
    model_path=config.paths.rnn_model_path,
    num_topics=config.num_labels,
    selected_layers_length=len(config.model.selected_layers),
    rnn_config=config.rnn,
)
```

### Files Created
- `{base_dir}/.embeddings/train/` - Extracted training representations
- `{base_dir}/.embeddings/val/` - Extracted validation representations  
- `{base_dir}/model/trained_model_rnn.pth` - Trained model weights
- `{base_dir}/model/checkpoints/` - Training checkpoints